# CLIP-H: discharge destination, DeepSeek-V4 arm

Predicts whether an adult hospital admission ends in a discharge **Home**, meaning a private residence without services, using information available at admission. Each prediction is explained with a small library of certified, human-readable clinical hypotheses. This notebook runs the DeepSeek-V4 arm of the paper. `discharge_runner_gpt.ipynb` runs the GPT-4.1 arm with the same settings and different models.

### Method
Each admission is rendered as a deterministic text profile and embedded with a local open-weight model (Qwen3-Embedding-0.6B). A Top-K sparse autoencoder (SAE) learns concept neurons from the embeddings, and the neurons most related to the outcome are selected with a diversity penalty (Phase 1). An LLM names each selected neuron, and names whose annotations do not reproduce the neuron's firing are discarded. The candidates are refined by a critic and a rewriter on a discovery split, screened by two LLM judges with majority voting, merged when they fire on the same patients, and certified on a held-out confirmation split by a prevalence floor, Benjamini-Hochberg FDR control and bootstrap stability (Phase 2). An L2-regularized logistic regression over the certified hypotheses makes the prediction, so every prediction decomposes into named concepts and their weights (Phase 3).

### Notebook layout
| Section | Content |
|---|---|
| Steps 1 to 6 | settings, compute check, paths, dependencies, API keys, seeding and library import |
| Preflight | configuration checks and one test call per model before any costly step |
| Data | reads the cohort, split and text profiles of `data/discharge/`, written by `mimic_cleaning/run/run_discharge.py` |
| Phase 1 | embeddings, SAE training, neuron selection |
| Phase 2.1 to 2.5 | interpretation and fidelity gate, self-refinement, clinical critique, deduplication, significance gate |
| Phase 3 | train and test annotation, logistic head, held-out AUROC and F1 with bootstrap intervals, descriptive refit, lineage |
| Evaluation | TF-IDF, structured logistic regression and XGBoost baselines on the same split |
| Run record | manifest with settings, models, package versions and results |

### Data
The cohort, the label, the split and the profiles are made by `mimic_cleaning/run/run_discharge.py` and written to `data/discharge/`; `mimic_cleaning/README.md` (section 8) describes them. This notebook reads them.

### Reproducibility
Every random step is seeded, and the embedding, SAE and logistic steps are deterministic. Interpretation and annotation run at temperature 0. DeepSeek calls use non-thinking mode, which honors the temperature setting. Hosted models are close to but not exactly deterministic, so a rerun from scratch can differ slightly.

### Before you run
1. Start Jupyter from this folder, the one that contains `mimic_eval/` and `myHypotheSAEs/`.
2. Write `data/discharge/` first: `python mimic_cleaning/run/run_discharge.py` (it reads MIMIC-IV v3.1 from `mimic_cleaning/mimic-iv-3.1/`; see `mimic_cleaning/README.md`).
3. Set `DEEPSEEK_API_KEY` and `ANTHROPIC_API_KEY` in the environment, or enter them when Step 5 asks.
4. Run all cells from the top. Every phase saves a checkpoint, so a rerun resumes where it stopped.


## Step 1: Settings
Every setting for a run is in the next cell, grouped by the phase that reads it. The values are the ones used in the paper. For a quick end-to-end check, set `N_PATIENTS_TRAIN, N_PATIENTS_TEST = 500, 100` here and in `mimic_cleaning/run/run_discharge.py` (rerun it), and `CHEAP_MODELS = True`. A changed setting gives the run its own folder in `clip_h_runs/`.


In [ ]:
# ============================== Settings ==============================
# Every setting for a run, grouped by the phase that reads it. The cells below need no edits.

TASK = 'discharge'
SEED = 42

# ---- Run --------------------------------------------------------------
RESUME       = True    # reuse the checkpoint of any stage already saved in RUN_DIR. RUN_DIR is
                       # named by a hash of these settings, so only an identical configuration
                       # reuses a checkpoint. Set False to recompute every stage.
CHEAP_MODELS = False   # True routes every DeepSeek role to deepseek-v4-flash for a quick check.
                       # The Claude judge is unchanged; set CRITIC_PROVIDER = 'openai' to drop it.

# ---- Models -----------------------------------------------------------
INTERP_MODEL = 'deepseek-v4-pro'            # Phase 2.1 neuron interpretation
CRITIC_MODEL = 'deepseek-v4-pro'            # Phase 2.2 critic and rewriter, Phase 2.3 judge
DEDUP_MODEL  = 'deepseek-v4-flash'          # Phase 2.4 deduplication adjudicator
CLAUDE_MODEL = 'claude-opus-4-8'            # Phase 2.3 second judge (needs ANTHROPIC_API_KEY)
ANNOT_MODEL  = 'deepseek-v4-flash'          # binary annotator for every feature matrix
EMB_MODEL    = 'Qwen/Qwen3-Embedding-0.6B'  # local open-weight embedder (Phase 1)
# DeepSeek limits concurrent requests rather than tokens per minute, so the worker counts set
# the request rate. Lower them if requests start to fail.
INTERP_WORKERS = 12                         # parallel interpretation requests
ANNOT_WORKERS  = 32                         # parallel annotation requests
USE_BATCH_ANNOTATION = False                # DeepSeek has no batch endpoint; keep False
if CHEAP_MODELS:
    INTERP_MODEL = CRITIC_MODEL = DEDUP_MODEL = ANNOT_MODEL = 'deepseek-v4-flash'
    print('[CHEAP_MODELS] every DeepSeek role uses deepseek-v4-flash (Claude unchanged)')

# ---- Data -------------------------------------------------------------
# data/discharge/ is written by mimic_cleaning/run/run_discharge.py. The cohort settings below must be the ones it
# was written with (its manifest.json); the Data cell stops if they differ.
N_PATIENTS_TRAIN = 5000         # training admissions after class balancing (None = full cohort)
N_PATIENTS_TEST  = 1000         # test admissions after class balancing (None = full cohort)
CACHE_NAME       = f'{TASK}_pipeline_s{SEED}_{N_PATIENTS_TRAIN}x{N_PATIENTS_TEST}'
                                # name of the annotation and embedding caches. The seed and the
                                # cohort sizes are part of it, so runs on different patients never
                                # share cached answers.
MIMIC_VERSION    = '3.1'        # MIMIC-IV version data/<task>/ was written from. The reported runs used
                                # '3.0' (same patients; BMI differs for ~40 of 6,000). '3.0' keeps their
                                # run folders; any other version gets its own.
SUBSAMPLE        = 'stratified' # seeded, label-stratified sample ('head' takes the first N rows)
TRAIN_BALANCE    = 0.5          # positive fraction of TRAIN after undersampling the majority class
TEST_BALANCE     = 0.5          # positive fraction of TEST (a balanced 500/500 evaluation set)
RICH_PROFILE     = True         # include the medication, culture and service sections of the profile
CLEAN_COHORT     = True         # discharge-cohort exclusions (facility admissions, excluded dispositions)

# ---- Phase 1: sparse autoencoder ---------------------------------------
SAE_M, SAE_K     = 1024, 16     # dictionary size and active neurons per patient
SAE_EPOCHS       = 1000         # maximum training epochs
SAE_PATIENCE     = 50           # early stopping on a 10% validation split of the training embeddings

# ---- Phase 2.1: neuron selection and interpretation ----------------------
N_HYPOTHESES     = 64           # neurons selected and interpreted (the candidate pool)
NAMING           = 'faithful'   # interpreter naming policy: 'faithful' names a neuron at the granularity
                                # of its firing; 'broad' prefers a broadly applicable label
SELECTION_METHOD = 'diverse'    # neuron selection: 'diverse' (MMR) or 'lasso'
MMR_LAMBDA       = 0.7          # MMR weight on outcome relevance versus redundancy
MIN_FIDELITY     = 0.30         # drop interpretations whose fidelity F1 is below this

# ---- Phase 2.2: self-refinement (Madaan et al., 2023) ------------------------
CONTROL_LOOP     = True         # run the critic and rewriter loop on the discovery split
REFINE_ITERS     = 3            # maximum refinement rounds
DISCOVERY_FRAC   = 0.7          # discovery share of TRAIN; the other 30% is the confirmation split

# ---- Phase 2.3: clinical critique ---------------------------------------
CRITIC_EVIDENCE_N = 300         # training profiles annotated to show the judges each prevalence
MIN_VALIDITY      = 1           # drop hypotheses with clinical validity below this (0 to 3)
LEAKAGE_DROP      = 2           # drop hypotheses with leakage risk at or above this (0 to 2)
CRITIC_PROVIDER   = 'both'      # judges: 'both' (CRITIC_MODEL and Claude), 'openai' (CRITIC_MODEL
                                # only) or 'anthropic' (Claude only)
CRITIC_VOTES      = 5           # votes per judge
CRITIC_VOTE_TEMP  = 0.5         # sampling temperature of the votes

# ---- Phase 2.4: deduplication by firing vectors ---------------------------
DEDUP_CORR_THR    = 0.90        # firing correlation (phi) at which a same-direction pair is considered
DEDUP_AUTO_THR    = 0.95        # phi at or above this merges automatically; pairs in
                                # [DEDUP_CORR_THR, DEDUP_AUTO_THR) go to the adjudicator, which can
                                # only veto the merge

# ---- Phase 2.5: significance gate (confirmation split) ---------------------
FDR_Q              = 0.10       # Benjamini-Hochberg false discovery rate
STABILITY_PI       = 0.6        # minimum bootstrap selection frequency
STABILITY_B        = 1000       # bootstrap resamples (on cached annotations, no API cost)
ESTIMABILITY_FLOOR = 15         # minimum number of firing patients
MIN_SURVIVORS      = 5          # if fewer hypotheses pass, the strongest by |effect| are added and
                                # marked provisional

# ---- Phase 3 and evaluation ---------------------------------------------
EVAL_N_BOOT       = 10000       # bootstrap resamples of the test set for every 95% interval
EVAL_ALPHA        = 0.05        # 95% percentile intervals
EVAL_RESAMPLE_BY  = 'row'       # bootstrap unit: 'row' (admission) or 'subject' (patient)
F1_THRESHOLD_MODE = 'prevalence_matched'   # F1 operating point: the k highest scores are positive,
                                # k = number of true positives, ties broken by a seeded permutation.
                                # It needs no calibration, so F1 is comparable across methods.
                                # Alternatives: 'train_tuned' and 'fixed' (0.5).
EXPORT_PREDICTIONS = True       # write the test-set predictions to predictions/

# ---- Start --------------------------------------------------------------
import time as _time
_PIPELINE_START = _time.time()
print(f'{TASK} pipeline | SAE M={SAE_M} K={SAE_K} | {N_HYPOTHESES} hypotheses | naming={NAMING} | seed={SEED}')
print(f'interp={INTERP_MODEL} annot={ANNOT_MODEL} critic={CRITIC_MODEL} dedup={DEDUP_MODEL} claude={CLAUDE_MODEL}')
print(f'bulk annotation: {"Batch API" if USE_BATCH_ANNOTATION else "synchronous"}')


## Step 2: Compute
Reports whether a CUDA GPU is available. The pipeline also runs on CPU, where embedding and SAE training are slower.


In [ ]:
# Silence known third-party notices that do not affect results: scikit-learn's migration of
# LogisticRegression from `penalty` to `l1_ratio`, and huggingface_hub's notice about
# unauthenticated downloads. Other warnings stay visible.
import warnings
for _msg in (r".*penalty.*deprecated.*", r".*l1_ratios.*", r".*use_legacy_attributes.*",
             r".*penalty=l1 with l1_ratio.*", r".*unauthenticated requests.*"):
    warnings.filterwarnings('ignore', message=_msg)

try:
    import torch
    if torch.cuda.is_available():
        print('Device: GPU -', torch.cuda.get_device_name(0))
    else:
        print('Device: CPU (embedding and SAE training are slower)')
except Exception as e:
    print('torch is not importable yet (it is installed in Step 4):', e)


## Step 3: Paths
`REPO_ROOT` is this task folder, found from the working directory. `DATA_DIR` is `data/discharge/` at the repository root, written by `mimic_cleaning/run/run_discharge.py`. Run outputs go to `clip_h_runs/` and embeddings to `emb_cache/`. Each line below should print `[OK]`.


In [ ]:
import os, sys

# REPO_ROOT is the nearest folder, from the working directory upward, that contains mimic_eval/.
REPO_ROOT = os.path.abspath('.')
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, 'mimic_eval')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)

# data/<task>/ at the repository root: the cohort, split and profiles written by
# mimic_cleaning/run/run_<task>.py.
DATA_DIR = os.path.normpath(os.path.join(REPO_ROOT, os.pardir, os.pardir, 'data', TASK))

MIMIC_EVAL = os.path.join(REPO_ROOT, 'mimic_eval')
HYP_LIB    = os.path.join(REPO_ROOT, 'myHypotheSAEs')
assert os.path.isdir(MIMIC_EVAL), f'mimic_eval not found under REPO_ROOT={REPO_ROOT}; start Jupyter in this task folder.'
os.chdir(MIMIC_EVAL)
for p in (HYP_LIB, MIMIC_EVAL):
    if p not in sys.path: sys.path.insert(0, p)
for p, lbl in [(DATA_DIR,f'data/{TASK}'),(MIMIC_EVAL,'mimic_eval'),(HYP_LIB,'myHypotheSAEs')]:
    print('[OK]' if os.path.isdir(p) else '[MISSING]', lbl, p)
RUN_BASE  = os.path.join(REPO_ROOT, 'clip_h_runs')
EMB_CACHE = os.path.join(REPO_ROOT, 'emb_cache')


## Step 4: Dependencies
Installs `requirements.txt` into the current Python environment. Skip this cell if the packages are already installed.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                os.path.join(REPO_ROOT, 'requirements.txt')], check=False)
print('Dependencies ready.')


## Step 5: API keys
Keys are read from the environment variables `DEEPSEEK_API_KEY` and `ANTHROPIC_API_KEY`. A key that is not set is asked for once and kept only for this session. DeepSeek runs every model role except the Claude judge, in non-thinking mode, which honors the temperature setting. Anthropic runs the Claude judge in Phase 2.3; set `CRITIC_PROVIDER = 'openai'` in Step 1 to use the DeepSeek judge alone.


In [ ]:
import os, getpass

def _api_key(name):
    """Return the key in environment variable `name`, asking for it once if it is not set."""
    value = os.environ.get(name) or getpass.getpass(f'{name}: ')
    os.environ[name] = value
    return value

# ---- DeepSeek: every model role except the Claude judge ----
DEEPSEEK_API_KEY = _api_key('DEEPSEEK_API_KEY')
assert DEEPSEEK_API_KEY, 'Set DEEPSEEK_API_KEY.'
os.environ['DEEPSEEK_BASE_URL'] = 'https://api.deepseek.com'
os.environ.setdefault('DEEPSEEK_THINKING', 'disabled')          # non-thinking mode honors temperature; a
                                                                # model id ending in '-think' opts that role in
os.environ.setdefault('DEEPSEEK_REASONING_EFFORT', 'medium')    # used only in thinking mode
os.environ['EMB_CACHE_DIR'] = EMB_CACHE
# DeepSeek limits concurrency rather than tokens per minute, so the token throttle is off and
# ANNOT_WORKERS and INTERP_WORKERS set the load.
os.environ.setdefault('OPENAI_TPM_LIMIT', '0')
print('DeepSeek configured. base_url =', os.environ['DEEPSEEK_BASE_URL'], '| thinking =', os.environ['DEEPSEEK_THINKING'])

# ---- Anthropic: the Claude judge in Phase 2.3 ----
if CRITIC_PROVIDER in ('anthropic', 'both'):
    _api_key('ANTHROPIC_API_KEY')
    print('Anthropic configured.')

# ---- Caching and retries ----
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
os.environ.setdefault('ANNOT_CHUNK', '1000')          # the annotation cache is saved every 1000 calls
os.environ.setdefault('ANNOT_RESCUE_ROUNDS', '4')     # sequential retry rounds for annotations that
os.environ.setdefault('ANNOT_RESCUE_WAIT_S', '60')    # failed in the parallel pass, and the wait between them


## Step 6: Seed and library
`L.set_global_seed` seeds Python, NumPy and PyTorch. `clip_h_lib`, imported as `L`, holds data loading, cohort construction, profiles, the curation helpers and evaluation. `RUN_DIR` is named by a hash of the settings and the seed, so a rerun with the same settings resumes and a different configuration gets its own folder.


In [ ]:
import json
import clip_h_lib as L
import numpy as np, pandas as pd
L.set_global_seed(SEED)
# data/<task>/manifest.json: the settings and column lists the data was written with. The
# signature of the task's feature and excluded-column lists is part of RUN_DIR, so a change to
# either list starts a new run folder.
_manifest_path = os.path.join(DATA_DIR, 'manifest.json')
if not os.path.exists(_manifest_path):
    raise FileNotFoundError(f'No {_manifest_path}: write data/{TASK}/ first with '
                            f'mimic_cleaning/run/run_{TASK}.py (see mimic_cleaning/README.md).')
with open(_manifest_path, encoding='utf-8') as _f:
    DATA_MANIFEST = json.load(_f)
_featsig = DATA_MANIFEST['featsig']
assert _featsig == L.config_hash({'feat': sorted(DATA_MANIFEST['feature_columns']),
                                  'forbid': sorted(DATA_MANIFEST['forbidden_columns'])}, 8), \
    'manifest.json: featsig does not match its column lists'
# Every setting that changes a checkpointed result is part of the hash, so a changed setting
# never reloads a checkpoint computed under the old value.
RUN_DIR = L.run_dir(RUN_BASE, TASK, {
    'task': TASK, 'interp': INTERP_MODEL, 'annot': ANNOT_MODEL, 'critic': CRITIC_MODEL,
    'dedup_model': DEDUP_MODEL, 'emb': EMB_MODEL, 'n_hyp': N_HYPOTHESES, 'selection': SELECTION_METHOD,
    'sae_m': SAE_M, 'sae_k': SAE_K, 'n_train': N_PATIENTS_TRAIN, 'n_test': N_PATIENTS_TEST,
    'rich': RICH_PROFILE, 'clean_cohort': CLEAN_COHORT, 'sample': SUBSAMPLE,
    'balance': (TRAIN_BALANCE, TEST_BALANCE), 'cheap_models': CHEAP_MODELS,
    'cloop': (CONTROL_LOOP, REFINE_ITERS, DISCOVERY_FRAC), 'naming': NAMING,
    'dedup': (DEDUP_CORR_THR, DEDUP_AUTO_THR),
    'gate': (FDR_Q, STABILITY_PI, STABILITY_B, ESTIMABILITY_FLOOR, MIN_SURVIVORS),
    'screen': (MIN_VALIDITY, LEAKAGE_DROP, CRITIC_PROVIDER, CRITIC_VOTES, CRITIC_VOTE_TEMP,
               CRITIC_EVIDENCE_N),
    'interp_knobs': (MIN_FIDELITY, MMR_LAMBDA), 'sae_train': (SAE_EPOCHS, SAE_PATIENCE),
    'pipeline': 'clip-h', 'featsig': _featsig,
    # The data version, left out for '3.0' so that the reported runs keep their folder names.
    **({} if MIMIC_VERSION == '3.0' else {'mimic_version': MIMIC_VERSION}),
}, SEED)
# Arm name in the predictions/ tree.
GENERATOR = 'deepseek' if 'deepseek' in str(INTERP_MODEL).lower() else 'gpt'
# Test-set predictions go to predictions/<task>/clip_h/<arm>.csv.
PRED_ROOT   = os.path.join(REPO_ROOT, 'predictions')
print('Run dir:', RUN_DIR)
print(f'Arm: {GENERATOR} | predictions -> {PRED_ROOT}')


## Preflight
Checks the paths, keys and settings, and sends one short request to every configured model, so a configuration error stops the run in seconds instead of hours into it.


In [ ]:
# Validate the configuration and send one short request to every configured model.
print('='*60); print('PREFLIGHT  |  configuration checks and one test call per model'); print('='*60)
for _lvl, _msg in L.preflight_config(dict(
        data_dir=DATA_DIR, critic_provider=CRITIC_PROVIDER,
        n_train=N_PATIENTS_TRAIN, n_test=N_PATIENTS_TEST,
        sae_m=SAE_M, sae_k=SAE_K, fdr_q=FDR_Q, stability_pi=STABILITY_PI)):
    print('  [' + _lvl + '] ' + _msg)

_fatal = []
for _m in dict.fromkeys([INTERP_MODEL, ANNOT_MODEL, CRITIC_MODEL, DEDUP_MODEL]):
    _ok, _msg = L.ping_model(_m)
    print(('  OK    ' if _ok else '  FAIL  ') + _msg)
    if not _ok:
        _fatal.append(_msg)
if CRITIC_PROVIDER in ('anthropic', 'both'):
    _ok, _msg = L.ping_model(CLAUDE_MODEL)
    print(('  OK    ' if _ok else '  FAIL  ') + _msg)
    # CRITIC_PROVIDER is part of RUN_DIR, so the run stops here rather than switching judges.
    if not _ok:
        _fatal.append(f"{_msg}  (CRITIC_PROVIDER={CRITIC_PROVIDER!r} needs the Claude judge; "
                      f"set CRITIC_PROVIDER = 'openai' in Step 1 to use one judge, "
                      f"then rerun from Step 1)")

if _fatal:
    raise RuntimeError('Preflight failed (no compute spent): ' + '  |  '.join(_fatal))
print('')
print('Preflight passed. Every phase saves a checkpoint in RUN_DIR, so a rerun resumes.')


## Data: read data/discharge/
Reads the train and test rows written by `mimic_cleaning/run/run_discharge.py` (`train_cohort.parquet`, `test_cohort.parquet`). The cell stops if `manifest.json` says they were written with other settings than Step 1, if an excluded column is present, or if a patient is in both train and test.

In [ ]:
import time as _time

print('='*60)
print(f'DATA  |  Read data/{TASK}/  (cohort, split and balance by mimic_cleaning/run/run_{TASK}.py)')
print('='*60)
_t0 = _time.time()

# The cohort settings of Step 1 are part of RUN_DIR, so they must be the ones data/<task>/ was
# written with. A difference stops the run here.
_want = {'seed': SEED, 'n_patients_train': N_PATIENTS_TRAIN, 'n_patients_test': N_PATIENTS_TEST,
         'subsample': SUBSAMPLE, 'train_balance': TRAIN_BALANCE, 'test_balance': TEST_BALANCE,
         'mimic_version': MIMIC_VERSION, 'rich_profile': RICH_PROFILE, 'clean_cohort': CLEAN_COHORT}
_differ = {k: (v, DATA_MANIFEST['settings'].get(k)) for k, v in _want.items()
           if DATA_MANIFEST['settings'].get(k) != v}
assert not _differ, (f'data/{TASK}/ was written with other settings (Step 1, data): {_differ}. '
                     f'Change Step 1, or rerun mimic_cleaning/run/run_{TASK}.py with these settings.')
train_df = pd.read_parquet(os.path.join(DATA_DIR, 'train_cohort.parquet'))
test_df  = pd.read_parquet(os.path.join(DATA_DIR, 'test_cohort.parquet'))
print(f'Read {DATA_DIR}  ({_time.time()-_t0:.1f}s)')

print(f"\nCohort: {DATA_MANIFEST['cohort_rows']} admissions | {train_df.shape[1]} columns")
print('Outcome balance in the cohort:', DATA_MANIFEST['label_counts_cohort'])
_forbidden = set(DATA_MANIFEST['forbidden_columns'])
_leaked = _forbidden & (set(train_df.columns) | set(test_df.columns))
assert not _leaked, f'Leakage guard: excluded {TASK} columns present in the data: {sorted(_leaked)}'
if _forbidden: print(f'Leakage guard OK: none of {len(_forbidden)} excluded columns present')

assert set(train_df['subject_id']) & set(test_df['subject_id']) == set(), 'a patient is in both train and test'
print(f'\nSplit  ->  Train {len(train_df)} | Test {len(test_df)} | patient-disjoint OK')
print(f'Train pos rate: {train_df["y"].mean():.3f} | Test pos rate: {test_df["y"].mean():.3f}')
if TRAIN_BALANCE is not None:
    _apt = SAE_K * len(train_df) / SAE_M
    _trp, _trn = int((train_df['y'] == 1).sum()), int((train_df['y'] == 0).sum())
    _tep, _ten = int((test_df['y'] == 1).sum()), int((test_df['y'] == 0).sum())
    print(f'  [balanced] TRAIN {_trp} pos / {_trn} neg ({train_df["y"].mean():.1%}) | TEST {_tep} pos / {_ten} neg ({test_df["y"].mean():.1%})')
    print('  [note] The balanced test set reports a 50/50 operating point, not the natural prevalence. '
          'AUROC does not depend on prevalence; F1, precision and Brier score describe the balanced set.')
    if abs(train_df['y'].mean() - 0.5) > 0.02 or abs(test_df['y'].mean() - 0.5) > 0.02:
        print('  [balanced] too few positives for an exact 50/50 split at this N; class_weight=balanced absorbs the rest.')
    print(f'  [balanced] SAE support ~{_apt:.0f} activations per neuron' + ('  (low: lower SAE_M or raise N)' if _apt < 25 else ''))
print(f'\nData complete in {_time.time()-_t0:.1f}s')


## Data: text profiles
Reads the profiles written by `mimic_cleaning/run/run_discharge.py` (`train_profile.jsonl`, `test_profile.jsonl`) and checks that they line up with the rows and labels. The run stops if train or test contains only one class; this can happen only with a very small `N_PATIENTS_*`.

In [ ]:
import time as _time

print('='*60)
print(f'DATA  |  Patient text profiles (data/{TASK}/)')
print('='*60)
_t0 = _time.time()
def _read_profiles(split):
    with open(os.path.join(DATA_DIR, f'{split}_profile.jsonl'), encoding='utf-8') as _f:
        return [json.loads(_l)['profile'] for _l in _f if _l.strip()]
train_profiles = _read_profiles('train')
test_profiles  = _read_profiles('test')
y_train = train_df['y'].to_numpy().astype(int)
y_test  = test_df['y'].to_numpy().astype(int)

for _split, _prof, _y in (('train', train_profiles, y_train), ('test', test_profiles, y_test)):
    _labels = pd.read_csv(os.path.join(DATA_DIR, f'{_split}_labels.csv'))[TASK].to_numpy()
    assert len(_prof) == len(_y) and np.array_equal(_labels, _y), f'{_split}: profiles, labels and rows do not line up'
assert len(np.unique(y_train)) >= 2 and len(np.unique(y_test)) >= 2, (
    'Train or test labels are single-class. Increase N_PATIENTS_* or use the full cohort '
    f'(N_PATIENTS_* = None), here and in mimic_cleaning/run/run_{TASK}.py.')

print(f'\nSample profile:\n{train_profiles[0][:400]}')
print(f'\nTrain pos rate {y_train.mean():.3f} | Test pos rate {y_test.mean():.3f}')
print(f'Profiles complete in {_time.time()-_t0:.1f}s')


## Phase 1: embeddings and sparse autoencoder
The local Qwen3-Embedding-0.6B model turns each profile into a 1024-dimensional vector, cached on disk. A Top-K SAE with `SAE_M = 1024` neurons and `SAE_K = 16` active per patient is trained on the training embeddings, with 10% of them held out for early stopping. The embedder runs locally because the SAE needs the model's internal representation, which hosted APIs do not expose. The SAE is saved in `RUN_DIR/sae/`. Neuron selection runs inside the next cell, as part of `generate_hypotheses`.


In [ ]:
import logging as _logging
_logging.getLogger("huggingface_hub").setLevel(_logging.ERROR)
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.quickstart import train_sae
import time as _time

print('='*60)
print('PHASE 1  |  Embed  ->  Train Top-K SAE')
print('='*60)
_t0 = _time.time()

print(f'Embedding {len(train_profiles)+len(test_profiles)} profiles with {EMB_MODEL}...')
all_texts = train_profiles + test_profiles
# chunk_size sets how often the embedding cache is written to disk. Embeddings are keyed by the
# exact profile text, so the chunk size does not change any value.
emb = get_local_embeddings(texts=all_texts, model=EMB_MODEL, cache_name=CACHE_NAME,
                           show_progress=True, chunk_size=500)
train_emb = np.stack([emb[t] for t in train_profiles])
test_emb  = np.stack([emb[t] for t in test_profiles])
L.validate_embeddings(train_emb, len(train_profiles)); L.validate_embeddings(test_emb, len(test_profiles))
print(f'Embeddings ready  train {train_emb.shape}  test {test_emb.shape}  ({_time.time()-_t0:.1f}s)')

print(f'\nTraining SAE  M={SAE_M}  K={SAE_K}  max_epochs={SAE_EPOCHS}  patience={SAE_PATIENCE}...')
_tt = _time.time()
L.set_global_seed(SEED)
# A seeded 10% of the training embeddings is held out for early stopping on reconstruction
# loss. Hypothesis generation in Phase 2 uses the activations of the full training set.
_rng = np.random.default_rng(SEED)
_perm = _rng.permutation(len(train_emb))
_n_val = max(1, int(0.1 * len(train_emb)))
sae_val_emb, sae_train_emb = train_emb[_perm[:_n_val]], train_emb[_perm[_n_val:]]
print(f'  SAE fit on {len(sae_train_emb)} embeddings, validating on {len(sae_val_emb)} for early stopping')
sae = train_sae(embeddings=sae_train_emb, val_embeddings=sae_val_emb, M=SAE_M, K=SAE_K,
                checkpoint_dir=os.path.join(RUN_DIR, 'sae'),
                n_epochs=SAE_EPOCHS, patience=SAE_PATIENCE, show_progress=True)
print(f'SAE trained in {_time.time()-_tt:.1f}s')
print(f'Phase 1 complete in {_time.time()-_t0:.1f}s')


## Phase 2: interpretation and curation
Phase 2 turns the SAE neurons into a certified hypothesis library. Each stage has one job and hands its output to the next.

```
2.1 INTERPRET    select neurons (MMR) -> LLM names each neuron -> fidelity gate (F1 >= 0.30)
2.2 REFINE       critic and rewriter loop: sharpen, split, re-ground, drop, broaden    (discovery split)
2.3 SCREEN       two judges x 5 votes score validity, leakage and direction; drop failures
2.4 DEDUP        merge concepts that fire on the same patients (phi), adjudicator veto  (discovery split)
2.5 GATE         prevalence floor -> BH-FDR -> bootstrap stability                       (confirmation split)
```

Stages 2.2 and 2.4 read only the discovery split (70% of TRAIN) and never read the outcome label. The gate tests significance once, on the disjoint confirmation split (the other 30%), so the multiple-testing correction is applied once.


## Phase 2.1: neuron selection, interpretation and fidelity gate
`generate_hypotheses` selects `N_HYPOTHESES` neurons by maximal marginal relevance, which trades outcome relevance against redundancy with the neurons already chosen. `INTERP_MODEL` writes three candidate names per neuron from profiles where the neuron fires strongly and profiles where it does not. The annotator then checks each candidate on a separate set of top- and zero-activation profiles, and the F1 between its answers and the neuron's firing is the candidate's fidelity. The best candidate is kept, and neurons whose best fidelity is below `MIN_FIDELITY` are dropped. Output: `hypotheses_raw.csv` and `stage_generate.csv`.


In [ ]:
from hypothesaes.quickstart import generate_hypotheses
import time as _time

print('='*60)
print('PHASE 2.1  |  Neuron selection -> LLM interpretation -> fidelity gate (F1 >= 0.30)')
print('='*60)
_t0 = _time.time()

TASK_INSTR = L.clinical_interpret_instructions(TASK)   # task framing for the interpreter: prefer clinical concepts, one standard granularity for demographics

hyp_ckpt = os.path.join(RUN_DIR, 'hypotheses_raw.csv')
if RESUME and os.path.exists(hyp_ckpt):
    hypotheses_df = pd.read_csv(hyp_ckpt)
    print(f'[cache] Loaded {len(hypotheses_df)} hypotheses from checkpoint  ({_time.time()-_t0:.1f}s)')
else:
    print(f'Selecting {N_HYPOTHESES} neurons (method={SELECTION_METHOD}), interpreting with {INTERP_MODEL}...')
    L.set_global_seed(SEED)
    hypotheses_df = generate_hypotheses(
        texts=train_profiles, labels=y_train, embeddings=train_emb, sae=sae,
        cache_name=CACHE_NAME, classification=True,
        selection_method=SELECTION_METHOD, n_selected_neurons=N_HYPOTHESES,
        interpreter_model=INTERP_MODEL, annotator_model=ANNOT_MODEL,
        n_examples_for_interpretation=20, n_scoring_examples=min(200, len(train_profiles)//4),
        n_candidate_interpretations=3, scoring_metric='f1',
        n_workers_interpretation=INTERP_WORKERS, n_workers_annotation=ANNOT_WORKERS,
        task_specific_instructions=TASK_INSTR, naming=NAMING, seed=SEED,
        mmr_lambda=MMR_LAMBDA,
        max_interpretation_tokens=3000,   # leaves room for the hidden reasoning tokens of reasoning models
        use_batch_scoring=USE_BATCH_ANNOTATION,
    )
    hypotheses_df.to_csv(hyp_ckpt, index=False)
    print(f'  Saved to {hyp_ckpt}  ({_time.time()-_t0:.1f}s)')

# Fidelity gate: drop interpretations whose annotations do not reproduce the neuron's firing.
L.validate_hypotheses(hypotheses_df)   # stops if generation returned no usable hypotheses
_fcol = [c for c in hypotheses_df.columns if 'fidelity' in c]
_fid = _fcol[0] if _fcol else None
if _fid and MIN_FIDELITY and MIN_FIDELITY > 0 and len(hypotheses_df) > 1:
    _mask = (hypotheses_df[_fid].fillna(0) >= MIN_FIDELITY).reset_index(drop=True)
    if int(_mask.sum()) < 1:                    # never drop the whole set
        _mask = pd.Series([True] * len(hypotheses_df))
else:
    _mask = pd.Series([True] * len(hypotheses_df))
# Stage log: every raw hypothesis, its fidelity F1 and whether the gate kept it.
_gen_rows = []
for _i, (_, _r) in enumerate(hypotheses_df.reset_index(drop=True).iterrows()):
    _kf = bool(_mask.iloc[_i])
    _gen_rows.append({'neuron_idx': _r.get('neuron_idx'), 'hypothesis': _r['interpretation'],
                      'fidelity_f1': (float(_r[_fid]) if _fid and pd.notna(_r[_fid]) else None),
                      'kept': _kf, 'reason': '' if _kf else f'fidelity<{MIN_FIDELITY}'})
L.log_stage_table(RUN_DIR, 'stage_generate', _gen_rows)
_before = len(hypotheses_df)
hypotheses_df = hypotheses_df[_mask.values].reset_index(drop=True)
if len(hypotheses_df) < _before:
    print(f'Fidelity gate (>= {MIN_FIDELITY} on {_fid}): kept {len(hypotheses_df)}/{_before}')

print(f'\n{len(hypotheses_df)} raw hypotheses:')
for i, r in hypotheses_df.iterrows():
    sc = [c for c in hypotheses_df.columns if 'fidelity' in c]
    print(f"  H{i+1:02d}: {r['interpretation']}" + (f"  (F1 {r[sc[0]]:.3f})" if sc else ''))
print(f'\nPhase 2.1 complete in {_time.time()-_t0:.1f}s')


## Phase 2.2: self-refinement (discovery split)
A critic and a rewriter improve the candidate set for up to `REFINE_ITERS` rounds (Self-Refine; Madaan et al., 2023). Each round annotates the discovery split, and hypotheses below 2% prevalence are marked for broadening. The critic sees each hypothesis and its prevalence, but not its association with the outcome, and returns one action per hypothesis: accept, sharpen, split, re-ground or drop. Drops are applied directly; the other actions become instructions for the rewriter. The loop stops early when nothing changes. Output: `ckpt_refine.csv`, `stage_refine.csv` and `self_refine_trace.json`.


In [ ]:
# Critic-in-the-loop refinement (Madaan et al., 2023, arXiv:2303.17651) on the discovery split.
# Prevalence alone marks rare hypotheses for broadening, and a clinical critic that never sees
# the outcome returns one action per hypothesis (accept, sharpen, split, re-ground, drop).
# Redundancy is handled in 2.4 and significance in 2.5.
import time as _time, json as _json
import self_refine as SR
from hypothesaes.annotate import annotate_texts_with_concepts
print('='*60); print('PHASE 2.2  |  Self-refinement on the discovery split'); print('='*60)
disc_mask = conf_mask = None
if not CONTROL_LOOP:
    print('  CONTROL_LOOP=False: skipped (the gate uses the full training set).')
else:
    _t0 = _time.time()
    disc_mask, conf_mask = SR.discovery_confirmation_split(len(train_profiles), frac=DISCOVERY_FRAC, seed=SEED, labels=np.asarray(y_train))
    assert not (disc_mask & conf_mask).any(), 'discovery and confirmation splits overlap'
    disc_profiles = [p for p, m in zip(train_profiles, disc_mask) if m]
    y_disc = np.asarray(y_train)[disc_mask]
    print(f'  split: discovery {int(disc_mask.sum())} | confirmation {int(conf_mask.sum())} (frac={DISCOVERY_FRAC})')
    def _annotate_disc(hs):
        # One annotation pass of the discovery split per round.
        return annotate_texts_with_concepts(texts=disc_profiles, concepts=list(hs),
            cache_name=CACHE_NAME + '_disc', model=ANNOT_MODEL, n_workers=ANNOT_WORKERS, seed=SEED,
            use_batch=USE_BATCH_ANNOTATION)
    _pos_label = {'discharge': 'Home', 'mortality': 'in-hospital death'}[TASK]
    _ref_ckpt = os.path.join(RUN_DIR, 'ckpt_refine.csv')
    if RESUME and os.path.exists(_ref_ckpt):
        hypotheses_df = pd.read_csv(_ref_ckpt); sr_trace = []
        print(f'  [resume] loaded {len(hypotheses_df)} refined hypotheses from {os.path.basename(_ref_ckpt)}')
    else:
        _refine = SR.make_refine_fn(CRITIC_MODEL, timeout=600, seed=SEED)
        _critic = SR.make_critic_fn(CRITIC_MODEL, task=TASK, pos_label=_pos_label, timeout=600, seed=SEED)
        _hyps0 = hypotheses_df['interpretation'].tolist()
        try:
            refined, sr_trace = SR.agentic_refine(_hyps0, y_disc, _annotate_disc, _refine,
                critic_fn=_critic, embed_fn=None, iters=REFINE_ITERS, min_prev=0.02)
            for tr in sr_trace:
                print(f"    iter {tr['iter']}: n={tr['n']} niche/broaden={tr['n_broadened']} "
                      f"critic-drop={tr['n_dropped']} critic-edit={tr['n_critic_edits']}")
            print(f'  Refinement: {len(_hyps0)} -> {len(refined)} hypotheses')
            # Stage log: per round and hypothesis, the prevalence, the broaden flag and the critic action.
            _ref_rows = [{'iter': tr['iter'], 'hypothesis': a['hypothesis'], 'prevalence': a['prevalence'],
                          'niche_broaden': a['niche_broaden'], 'critic_action': a['critic_action'],
                          'critic_feedback': a['critic_feedback']}
                         for tr in sr_trace for a in tr.get('actions', [])]
            if _ref_rows:
                L.log_stage_table(RUN_DIR, 'stage_refine', _ref_rows)
            if refined:
                hypotheses_df = pd.DataFrame({'interpretation': refined})
        except Exception as _e:
            print(f'  [warn] refinement failed ({_e}); keeping the unrefined hypothesis set.'); sr_trace = []
        _json.dump({'trace': sr_trace, 'discovery_n': int(disc_mask.sum()), 'confirmation_n': int(conf_mask.sum())},
                   open(os.path.join(RUN_DIR, 'self_refine_trace.json'), 'w'), indent=2)
        hypotheses_df.to_csv(_ref_ckpt, index=False)
    print(f'  Phase 2.2 complete in {_time.time()-_t0:.1f}s ({len(hypotheses_df)} hypotheses forwarded)')


## Phase 2.3: clinical critique
Two judges, `CRITIC_MODEL` and Claude, each vote `CRITIC_VOTES` times on every hypothesis: expected direction and magnitude, clinical validity (0 to 3), leakage risk (0 to 2), a demographic tag and coherence. Judges see each hypothesis's prevalence but not its association with the outcome. Votes are aggregated within each judge and then combined across judges. Hypotheses with leakage risk 2, validity below 1, or that conflate distinct clinical signals are dropped. Output: `hypotheses_final.csv`, `critic_evidence.csv`, `stage_screen.csv` and the saved votes in `screen_votes/`.


In [ ]:
# Clinical critique as a two-stage self-consistency vote. Each judge votes CRITIC_VOTES times,
# independently, on every axis (direction, magnitude, validity, leakage, demographic, coherence).
# Votes are aggregated within each judge, then combined across judges: leakage takes the maximum,
# demographic the OR, coherence the AND, and direction the shared value if the judges agree,
# otherwise 'unclear'. Judges see prevalence but not the outcome association, so the expected
# direction is an a-priori judgment.
import json as _json, re as _re, time as _time, hashlib as _hashlib
import numpy as np
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.llm_api import get_completion
print('='*60); print('PHASE 2.3  |  Clinical critique: two-stage self-consistency vote'); print('='*60)
_t0 = _time.time()
_in = hypotheses_df['interpretation'].tolist()
_pos_label = {'discharge': 'Home', 'mortality': 'in-hospital death'}[TASK]

def _gi(d, k, dv=0):
    try: return int(d.get(k, dv))
    except Exception: return dv

# Prevalence on a training subsample is shown to the judges; the outcome association is not.
prevalence = {h: float('nan') for h in _in}
try:
    _n_ev = min(int(CRITIC_EVIDENCE_N), len(train_profiles))
    _ev = annotate_texts_with_concepts(texts=train_profiles[:_n_ev], concepts=_in,
        cache_name=CACHE_NAME + '_train_final', model=ANNOT_MODEL, n_workers=ANNOT_WORKERS, seed=SEED,
        use_batch=USE_BATCH_ANNOTATION)  # shares its cache with the Phase 3 training annotations
    prevalence = {h: float(np.mean(np.asarray(_ev[h], dtype=float) != 0)) for h in _in}
    pd.DataFrame([{'hypothesis': h, 'prevalence': prevalence[h]} for h in _in]).to_csv(
        os.path.join(RUN_DIR, 'critic_evidence.csv'), index=False)
except Exception as _e:
    print(f'  [warn] prevalence grounding unavailable ({_e}); judges see no prevalence.')

_judges = []
if CRITIC_PROVIDER in ('openai', 'both'):    _judges.append(('gpt', CRITIC_MODEL))
if CRITIC_PROVIDER in ('anthropic', 'both'): _judges.append(('claude', CLAUDE_MODEL))
if not _judges: _judges = [('gpt', CRITIC_MODEL)]

_INSTRUMENT_GROUNDING = L.clinical_instrument_grounding(TASK)  # validated clinical scores the judges may cite for mechanism and validity
SCREEN_SYSTEM = (
    'You are a senior clinical-informatics scientist auditing machine-generated hypotheses that will '
    f'become features predicting the {TASK} outcome (positive class: {_pos_label}). You are shown each '
    'hypothesis and its prevalence, but NOT its association with the outcome, so your direction and '
    'validity calls stay honest a-priori judgments. '
    + (_INSTRUMENT_GROUNDING + ' ' if _INSTRUMENT_GROUNDING else '')
    + 'Respond with only valid JSON.')
_numbered = '\n'.join(f'{i}: "{h}"  [prevalence={"n/a" if prevalence.get(h)!=prevalence.get(h) else f"{prevalence[h]*100:.0f}%"}]'
                       for i, h in enumerate(_in))
SCREEN_PROMPT = (
    'For every hypothesis id return: expected_direction (increase|decrease|unclear) of its effect on '
    f'P({_pos_label}); expected_magnitude (none|weak|moderate|strong); clinical_validity (0-3: 0 noise, '
    '1 weak, 2 plausible, 3 strong/well-established); is_demographic (a neutral transparency tag, NOT a penalty -- true if the concept is defined mainly '
    'by a demographic attribute such as age band, sex, race, insurance, or language; do not lower clinical_validity for it); leakage_risk (0 none, 1 possible '
    'circularity, 2 direct outcome leakage/tautology); coherent (false if it conflates two or more '
    'clinically distinct signals); mechanism (one-sentence clinical reason).\n'
    'JSON: {"judgments":[{"id":<int>,"expected_direction":"...","expected_magnitude":"...",'
    '"clinical_validity":<int>,"is_demographic":<bool>,"leakage_risk":<int>,"coherent":<bool>,'
    '"mechanism":"..."}, ...]}\n\n<hypotheses>\n' + _numbered + '\n</hypotheses>')

# Each parsed vote is saved to RUN_DIR/screen_votes as soon as it returns, keyed by model,
# hypothesis-set signature and vote index. An interrupted run resumes from the saved votes, and a
# changed hypothesis set gets a new signature, so stale votes are never reused.
_vote_dir = os.path.join(RUN_DIR, 'screen_votes'); os.makedirs(_vote_dir, exist_ok=True)
_setsig = _hashlib.sha256('\x1e'.join(_in).encode('utf-8')).hexdigest()[:8]
# Retries per vote, configurable through the environment.
_VOTE_TRIES = int(os.environ.get('SCREEN_VOTE_TRIES', '3'))
_VOTE_WAIT_S = float(os.environ.get('SCREEN_VOTE_WAIT_S', '60'))
_VOTES_LOST = []

def _run_judge(model, n_votes):
    acc = {i: {'direction': [], 'magnitude': [], 'validity': [], 'demographic': [],
               'leakage': [], 'coherent': [], 'mechanism': []} for i in range(len(_in))}
    _safe = _re.sub(r'[^A-Za-z0-9._-]', '_', str(model))
    for v in range(int(n_votes)):
        _vf = os.path.join(_vote_dir, f'{_safe}__{_setsig}__v{v}.json')
        _judgments = None
        if RESUME and os.path.exists(_vf):
            try: _judgments = _json.load(open(_vf, encoding='utf-8'))
            except Exception: _judgments = None
        if _judgments is None:
            # get_completion retries transient errors itself; these outer attempts cover a longer
            # outage, because a lost vote removes a member of the self-consistency ensemble.
            for _vt in range(_VOTE_TRIES):
                try:
                    _r = get_completion(prompt=SCREEN_PROMPT, model=model, system=SCREEN_SYSTEM,
                                        max_tokens=6000, timeout=600, seed=SEED + v, temperature=CRITIC_VOTE_TEMP)
                    _s = _re.sub(r'^```(?:json)?|```$', '', str(_r).strip(), flags=_re.MULTILINE).strip()
                    _m = _re.search(r'\{.*\}', _s, _re.DOTALL)
                    _judgments = _json.loads(_m.group(0) if _m else _s)['judgments']
                    _json.dump(_judgments, open(_vf, 'w', encoding='utf-8'))   # checkpoint this vote
                    break
                except Exception as _e:
                    print(f'  [{model} vote {v+1}/{n_votes}] attempt {_vt+1}/{_VOTE_TRIES} failed: {_e}')
                    if _vt < _VOTE_TRIES - 1:
                        _time.sleep(_VOTE_WAIT_S)
            if _judgments is None:
                _VOTES_LOST.append((model, v)); continue
        for j in _judgments:
            i = int(j['id'])
            if i in acc:
                acc[i]['direction'].append(str(j.get('expected_direction', 'unclear')).lower())
                acc[i]['magnitude'].append(str(j.get('expected_magnitude', 'moderate')).lower())
                acc[i]['validity'].append(_gi(j, 'clinical_validity', 2))
                acc[i]['demographic'].append(bool(j.get('is_demographic', False)))
                acc[i]['leakage'].append(_gi(j, 'leakage_risk', 0))
                acc[i]['coherent'].append(bool(j.get('coherent', True)))
                acc[i]['mechanism'].append(str(j.get('mechanism', '')))
    return acc

print(f'Voting: {[m for _, m in _judges]} x {CRITIC_VOTES} self-consistency samples on {len(_in)} hypotheses...')
_by_model = {name: _run_judge(model, CRITIC_VOTES) for name, model in _judges}
_axes = ('direction', 'magnitude', 'validity', 'demographic', 'leakage', 'coherent')
rows = []
_dir_by_model = {name: [] for name in _by_model}
for i, h in enumerate(_in):
    votes = {name: {ax: _by_model[name][i][ax] for ax in _axes}
             for name in _by_model if _by_model[name][i]['direction']}
    verdict = L.two_stage_screen_vote(votes)
    _mechs = [m for name in _by_model for m in _by_model[name][i]['mechanism'] if m]
    verdict['mechanism'] = _mechs[0] if _mechs else ''
    verdict['interpretation'] = h
    verdict['source_ids'] = [i]
    rows.append(verdict)
    for name in _by_model:
        _d = _by_model[name][i]['direction']
        _dir_by_model[name].append(max(set(_d), key=_d.count) if _d else 'unclear')
# If a judge lost the majority of its votes, stop instead of screening on the remainder. Saved
# votes are reused, so rerunning this cell repeats only the lost calls.
if _VOTES_LOST:
    from collections import Counter as _VC
    _lost_by = _VC(m for m, _ in _VOTES_LOST)
    print(f'  [warn] {len(_VOTES_LOST)} SCREEN vote(s) lost after {_VOTE_TRIES} attempts: {dict(_lost_by)}')
    _crippled = {m: c for m, c in _lost_by.items() if c > CRITIC_VOTES // 2}
    if _crippled:
        raise RuntimeError(
            f'SCREEN lost the majority of votes for {_crippled} (of {CRITIC_VOTES} per judge). '
            f'Fix the connection and re-run this cell; checkpointed votes are not re-paid.')
ensemble_kappa = None
if len([n for n in _dir_by_model if _by_model[n][0]['direction']]) >= 2:
    _a, _b = list(_dir_by_model.values())[:2]
    try: ensemble_kappa = round(L.cohen_kappa(_a, _b), 3)
    except Exception: ensemble_kappa = None

# Drop hypotheses with leakage >= LEAKAGE_DROP or validity < MIN_VALIDITY, and those that conflate
# distinct clinical signals (coherent = false).
kept = [r for r in rows if r['leakage_risk'] < LEAKAGE_DROP and r['clinical_validity'] >= MIN_VALIDITY and r['coherent']]
# A hypothesis without a parseable vote is dropped, and an empty screen stops the run: it
# indicates a judge outage or malformed responses, not a set of failing hypotheses.
_n_screen_failed = sum(1 for r in rows if r.get('screen_failed'))
if _n_screen_failed:
    print(f'  [warn] {_n_screen_failed}/{len(rows)} concepts had no parseable judge vote and were '
          f'dropped.')
predict = kept
if not predict:
    raise RuntimeError(
        f'SCREEN kept 0 of {len(rows)} concepts '
        f'(leakage>={LEAKAGE_DROP} / validity<{MIN_VALIDITY} / incoherent; '
        f'{_n_screen_failed} had no parseable vote). Check the judge responses and rerun this cell.')
_ncols = ['interpretation', 'is_demographic', 'clinical_validity', 'leakage_risk',
          'expected_direction', 'expected_magnitude', 'critic_confidence', 'critic_disagreement',
          'mechanism', 'source_ids']
hypotheses_df = pd.DataFrame([{c: r[c] for c in _ncols} for r in predict])
hypotheses_df.to_csv(os.path.join(RUN_DIR, 'hypotheses_final.csv'), index=False)
# Stage log: every screened concept with its verdict and the reason it was kept or dropped.
_pset = {r['interpretation'] for r in predict}
_screen_rows = []
for r in rows:
    _k = r['interpretation'] in _pset
    if _k: _rsn = ''
    elif r['leakage_risk'] >= LEAKAGE_DROP: _rsn = f'leakage_risk>={LEAKAGE_DROP}'
    elif r['clinical_validity'] < MIN_VALIDITY: _rsn = f'validity<{MIN_VALIDITY}'
    elif not r['coherent']: _rsn = 'incoherent'
    else: _rsn = 'dropped'
    _screen_rows.append({'hypothesis': r['interpretation'], 'clinical_validity': r['clinical_validity'],
                         'expected_direction': r['expected_direction'], 'leakage_risk': r['leakage_risk'],
                         'is_demographic': r['is_demographic'], 'coherent': r['coherent'],
                         'critic_confidence': r.get('critic_confidence'),
                         'critic_disagreement': r.get('critic_disagreement'), 'kept': _k, 'reason': _rsn})
L.log_stage_table(RUN_DIR, 'stage_screen', _screen_rows)
_ndrop = len(rows) - len(predict); _ndemo = int(hypotheses_df['is_demographic'].sum())
_ndis = int(hypotheses_df['critic_disagreement'].sum())
print(f'  {len(rows)} screened -> {len(predict)} kept (dropped {_ndrop} leaky/low-validity/incoherent) | '
      f'{_ndemo} demographic | {_ndis} cross-model disagreement' + (f' | kappa={ensemble_kappa}' if ensemble_kappa is not None else ''))
for _, r in hypotheses_df.iterrows():
    _fl = '  <DISAGREE>' if r['critic_disagreement'] else ''
    print(f"  [v{int(r['clinical_validity'])} {str(r['expected_direction']):>8} conf {r['critic_confidence']:.0%}]"
          f"{' [demographic]' if r['is_demographic'] else ''} {r['interpretation'][:52]}{_fl}")
print(f'Phase 2.3 complete in {_time.time()-_t0:.1f}s')

## Phase 2.4: deduplication by firing vectors (discovery split)
Redundancy is judged on the data, not on the wording: two concepts are redundant when they fire on the same patients. Each screened concept is annotated on the discovery split, and the firing correlation `phi` is computed for every pair. A same-direction pair with `phi >= DEDUP_AUTO_THR` merges automatically; a pair in `[DEDUP_CORR_THR, DEDUP_AUTO_THR)` goes to `DEDUP_MODEL`, which can only veto the merge; pairs with opposite expected directions never merge. A merged feature is the OR of its members' annotations. Output: `dedup_pairs.csv`, `dedup_clusters.csv`, `dedup_complements.csv` and `dedup_map.csv`.


In [ ]:
# Redundancy is judged on firing vectors on the discovery split, not on wording. Two concepts merge
# only if they fire on nearly the same patients (phi) and share an expected direction:
# phi >= DEDUP_AUTO_THR merges automatically, [DEDUP_CORR_THR, DEDUP_AUTO_THR) goes to an LLM
# adjudicator that can only veto the merge, and lower phi keeps them apart. A merged feature is the
# OR of its members' cached annotations.
import time as _time
from hypothesaes.annotate import annotate_texts_with_concepts
import json as _json
print('='*60); print('PHASE 2.4  |  Deduplication by firing vectors (phi)'); print('='*60)
_t0 = _time.time()
_dd_ckpt = os.path.join(RUN_DIR, 'ckpt_dedup.csv'); _dd_map_ckpt = os.path.join(RUN_DIR, 'ckpt_dedup_map.json')
if RESUME and os.path.exists(_dd_ckpt) and os.path.exists(_dd_map_ckpt):
    hypotheses_df = pd.read_csv(_dd_ckpt)
    MERGE_SOURCE_MAP = _json.load(open(_dd_map_ckpt, encoding='utf-8'))
    print(f'  [resume] loaded {len(hypotheses_df)} deduplicated concepts and the merge map from the checkpoint')
else:
    _labels = hypotheses_df['interpretation'].tolist()
    _dirs = (hypotheses_df['expected_direction'].tolist() if 'expected_direction' in hypotheses_df.columns
             else ['unclear'] * len(_labels))
    _meta = hypotheses_df.to_dict('records')
    _pos_label = {'discharge': 'Home', 'mortality': 'in-hospital death'}[TASK]
    # Firing vectors on the discovery split, sharing the refinement cache. Without refinement a
    # training subsample is used.
    if CONTROL_LOOP and ('disc_profiles' in dir()) and disc_profiles:
        _dd_profiles, _dd_cache = disc_profiles, CACHE_NAME + '_disc'
    else:
        _dd_profiles = train_profiles[:min(1000, len(train_profiles))]; _dd_cache = CACHE_NAME + '_train_final'
    def _dd_annotate(_texts):
        return annotate_texts_with_concepts(texts=_dd_profiles, concepts=list(_texts),
            cache_name=_dd_cache, model=ANNOT_MODEL, n_workers=ANNOT_WORKERS, seed=SEED, use_batch=USE_BATCH_ANNOTATION)
    if len(_labels) > 1:
        _dd_ann = _dd_annotate(_labels)
        _vecs = [np.asarray(_dd_ann[h], dtype=float) for h in _labels]
        _adj = L.make_dedup_adjudicator(DEDUP_MODEL, task=TASK, pos_label=_pos_label, timeout=300, seed=SEED)
        DEDUP_RESULT = L.predictive_dedup(_vecs, _labels, _dirs, meta=_meta,
            corr_thr=DEDUP_CORR_THR, auto_thr=DEDUP_AUTO_THR, adjudicate_fn=_adj)
    else:
        DEDUP_RESULT = {'source_map': {}, 'kept': list(_labels), 'pairs': [], 'clusters': []}
    MERGE_SOURCE_MAP = DEDUP_RESULT['source_map']
    _kept = DEDUP_RESULT['kept']
    # Stage logs: every considered pair (phi, tier, decision), every merge cluster and the source map.
    L.log_stage_table(RUN_DIR, 'dedup_pairs',
        DEDUP_RESULT['pairs'] or [{'note': 'no pair reached DEDUP_CORR_THR'}])
    L.log_stage_table(RUN_DIR, 'dedup_clusters',
        [{'representative': c['representative'], 'members': ' | '.join(c['members']),
          'phi_to_rep': '; '.join(f'{k}={v}' for k, v in c['phi_to_rep'].items()),
          'rep_prevalence': c.get('rep_prevalence'), 'union_prevalence': c.get('union_prevalence'),
          'rep_covers_union': c.get('rep_covers_union')} for c in DEDUP_RESULT['clusters']]
        or [{'representative': '', 'members': '', 'phi_to_rep': '',
             'rep_prevalence': '', 'union_prevalence': '', 'rep_covers_union': ''}])
    # Complementary pairs (phi <= -DEDUP_CORR_THR) fire on disjoint patients, so one is the negation
    # of the other ("male sex" and "female sex"). Keeping both would put collinear columns in the
    # design matrix and count one variable twice, so one side of each near-perfect pair is dropped.
    L.log_stage_table(RUN_DIR, 'dedup_complements',
        DEDUP_RESULT.get('complements') or [{'note': 'no complementary pair reached -DEDUP_CORR_THR'}])
    for _drop, _keep in (DEDUP_RESULT.get('dropped_complement') or {}).items():
        print(f'    dropped complement: "{_drop}"  (kept its complement "{_keep}")')
    # A merged feature is the OR of its members, so its label should describe that union.
    # rep_covers_union is the share of union firings covered by the representative alone.
    _bad_rep = [c for c in DEDUP_RESULT['clusters'] if (c.get('rep_covers_union') or 1.0) < 0.98]
    if _bad_rep:
        print(f'  [warn] {len(_bad_rep)} cluster(s) whose representative label describes less than 98% of '
              f'the OR-union it is fit on -- inspect dedup_clusters.csv:')
        for c in _bad_rep:
            print(f'      cover={c["rep_covers_union"]:.2f}  rep="{c["representative"]}"  '
                  f'union_prev={c["union_prevalence"]}')
    pd.DataFrame([{'representative': r, 'sources': ' | '.join(MERGE_SOURCE_MAP.get(r, [r]))} for r in _kept]).to_csv(
        os.path.join(RUN_DIR, 'dedup_map.csv'), index=False)
    # Keep the representatives and unmerged concepts, with their screening metadata.
    hypotheses_df = (hypotheses_df[hypotheses_df['interpretation'].isin(_kept)]
                     .drop_duplicates('interpretation').reset_index(drop=True))
    # A merged feature takes the most conservative flags of its members (OR of is_demographic,
    # maximum leakage_risk), so a merge never hides a demographic or leaky member.
    _demo_by = {r['interpretation']: bool(r.get('is_demographic', False)) for r in _meta}
    _leak_by = {r['interpretation']: int(r.get('leakage_risk', 0) or 0) for r in _meta}
    for _rep, _members in MERGE_SOURCE_MAP.items():
        _rm = hypotheses_df['interpretation'] == _rep
        if 'is_demographic' in hypotheses_df.columns:
            hypotheses_df.loc[_rm, 'is_demographic'] = any(_demo_by.get(m, False) for m in _members)
        if 'leakage_risk' in hypotheses_df.columns:
            hypotheses_df.loc[_rm, 'leakage_risk'] = max([_leak_by.get(m, 0) for m in _members] + [0])
    print(f'  {len(_labels)} screened -> {len(_kept)} after firing-vector dedup '
          f'({len(DEDUP_RESULT["clusters"])} merge group(s))')
    for c in DEDUP_RESULT['clusters']:
        _others = [m for m in c['members'] if m != c['representative']]
        print(f'    merged -> "{c["representative"]}"  <=  ' + ' + '.join(f'"{m}"' for m in _others))
    for p in DEDUP_RESULT['pairs']:
        if p.get('tier') == 'llm_vetoed_kept':
            print(f'    kept apart (phi={p["phi"]}, llm-veto): "{p["concept_a"]}" vs "{p["concept_b"]}" -- {p["llm_reason"]}')
    # hypotheses_final.csv holds the deduplicated set from here on.
    hypotheses_df.to_csv(os.path.join(RUN_DIR, 'hypotheses_final.csv'), index=False)
    hypotheses_df.to_csv(_dd_ckpt, index=False)   # checkpoint for resume
    _json.dump(MERGE_SOURCE_MAP, open(_dd_map_ckpt, 'w', encoding='utf-8'))
print(f'Phase 2.4 complete in {_time.time()-_t0:.1f}s')

## Phase 2.5: significance gate (confirmation split)
The deduplicated concepts are annotated on the confirmation split, which self-refinement and deduplication do not use. A concept is certified only if it fires in at least `ESTIMABILITY_FLOOR` patients, is significant under Benjamini-Hochberg control at `FDR_Q`, and is selected in at least `STABILITY_PI` of `STABILITY_B` bootstrap resamples. The multiple-testing correction is applied once, here. Output: `hypotheses_stable.csv` and `stage_gate.csv`.


In [ ]:
# Significance gate on the confirmation split. A hypothesis is certified only if it (i) fires in at
# least ESTIMABILITY_FLOOR patients, (ii) is significant under Benjamini-Hochberg control at FDR_Q,
# and (iii) is selected in at least STABILITY_PI of STABILITY_B bootstrap resamples. If fewer than
# MIN_SURVIVORS pass, the strongest remaining hypotheses by |effect| are added as provisional.
from hypothesaes.annotate import annotate_texts_with_concepts
import time as _time

print('='*60); print('PHASE 2.5  |  Significance gate: prevalence floor, BH-FDR, bootstrap stability'); print('='*60)
_t0 = _time.time()
_gate_ckpt = os.path.join(RUN_DIR, 'hypotheses_stable.csv')   # the gate output is also its checkpoint
if RESUME and os.path.exists(_gate_ckpt):
    stable_df = pd.read_csv(_gate_ckpt)
    stable_df['keep'] = stable_df['keep'].fillna(False).astype(bool)
    if 'provisional' not in stable_df.columns: stable_df['provisional'] = False
    stable_df['provisional'] = stable_df['provisional'].fillna(False).astype(bool)
    hyps_stable = stable_df.loc[stable_df['keep'] | stable_df['provisional'], 'hypothesis'].tolist()
    GATE_MIN_FIRE = int(ESTIMABILITY_FLOOR)
    provisional = stable_df.loc[stable_df['provisional'], 'hypothesis'].tolist()
    uni_map = dict(zip(stable_df['hypothesis'], stable_df['univariate_direction'])) if 'univariate_direction' in stable_df.columns else {}
    sig_map = dict(zip(stable_df['hypothesis'], (stable_df['significant'].fillna(False) if 'significant' in stable_df.columns else [False]*len(stable_df))))
    MERGE_SOURCE_MAP = globals().get('MERGE_SOURCE_MAP', {})
    print(f'  [resume] loaded gate results from {os.path.basename(_gate_ckpt)}: {len(hyps_stable)} passed to Phase 3')
else:
    _hyps = hypotheses_df['interpretation'].tolist()
    # With self-refinement on, significance is tested on the confirmation split, which is disjoint
    # from the discovery split used to refine and deduplicate.
    if CONTROL_LOOP and ('conf_mask' in dir()) and conf_mask is not None:
        assert not (disc_mask & conf_mask).any(), 'discovery and confirmation splits overlap'
        _gate_profiles = [p for p, m in zip(train_profiles, conf_mask) if m]
        _gate_y = np.asarray(y_train)[conf_mask]; _gate_cache = CACHE_NAME + '_conf'
        print(f'Annotating {len(_gate_profiles)} confirmation profiles x {len(_hyps)} hypotheses for the gate...')
    else:
        _gate_profiles, _gate_y, _gate_cache = train_profiles, np.asarray(y_train), CACHE_NAME + '_train_final'
        print(f'Annotating {len(_gate_profiles)} train profiles x {len(_hyps)} hypotheses for gating...')
    # A merged feature from Phase 2.4 is the OR of its members' annotations; other concepts are
    # annotated directly.
    MERGE_SOURCE_MAP = globals().get('MERGE_SOURCE_MAP', {})
    def _gate_annotate(_texts):
        return annotate_texts_with_concepts(texts=_gate_profiles, concepts=list(_texts),
            cache_name=_gate_cache, model=ANNOT_MODEL, n_workers=ANNOT_WORKERS, seed=SEED,
            use_batch=USE_BATCH_ANNOTATION)
    train_ann_gate = L.annotate_features(_gate_profiles, _hyps, MERGE_SOURCE_MAP, _gate_annotate)

    _minfire = ESTIMABILITY_FLOOR
    stable_df = L.stable_hypotheses(train_ann_gate, _gate_y, n_splits=5, seed=SEED, n_boot=1000,
                                    min_fire=_minfire, fdr_q=FDR_Q, stability_pi=STABILITY_PI,
                                    stability_B=STABILITY_B)
    stable_df = (hypotheses_df[['interpretation']].rename(columns={'interpretation': 'hypothesis'})
                 .merge(stable_df, on='hypothesis', how='left'))
    stable_df['keep'] = stable_df['keep'].fillna(False)

    hyps_stable = [h for h, k in zip(stable_df['hypothesis'], stable_df['keep']) if k]
    provisional = []
    if len(hyps_stable) < MIN_SURVIVORS:        # keep Phase 3 from receiving an empty or tiny set
        extra = (stable_df[~stable_df['keep']]
                 .assign(_abs=stable_df['univariate_coef'].abs())
                 .sort_values('_abs', ascending=False)['hypothesis']
                 .head(MIN_SURVIVORS - len(hyps_stable)).tolist())
        provisional = extra
        hyps_stable = hyps_stable + extra
        print(f'  [safety floor] only {len(hyps_stable)-len(extra)} passed; added {len(extra)} '
              f'provisional (top-|effect|) to reach MIN_SURVIVORS={MIN_SURVIVORS}.')
    stable_df['provisional'] = stable_df['hypothesis'].isin(provisional)

    uni_map = dict(zip(stable_df['hypothesis'], stable_df['univariate_direction']))
    sig_map = dict(zip(stable_df['hypothesis'], stable_df['significant'].fillna(False)))
    stable_df.to_csv(os.path.join(RUN_DIR, 'hypotheses_stable.csv'), index=False)
    # Stage log: per hypothesis, the firing count, q-value, selection frequency and the failed criterion.
    _gate_rows = []
    for _, r in stable_df.iterrows():
        if bool(r['keep']): _st, _cr = 'pass', ''
        elif bool(r.get('provisional', False)): _st, _cr = 'provisional', 'safety-floor top-|effect|'
        else:
            _st = 'fail'; _nf = r['n_fire'] if r['n_fire'] == r['n_fire'] else 0
            if _nf < _minfire: _cr = f'prevalence<{_minfire}'
            elif not bool(r.get('significant', False)): _cr = f'FDR q>{FDR_Q}'
            else: _cr = f'stability<{STABILITY_PI}'
        _gate_rows.append({'hypothesis': r['hypothesis'], 'n_fire': r['n_fire'],
                           'univariate_q': r.get('qvalue'), 'selection_frequency': r.get('selection_frequency'),
                           'status': _st, 'failed_criterion': _cr})
    L.log_stage_table(RUN_DIR, 'stage_gate', _gate_rows)

    GATE_MIN_FIRE = int(_minfire)   # recorded in the manifest
    print(f'  min firing patients = {_minfire} | FDR q <= {FDR_Q} | stability pi >= {STABILITY_PI}')
    print(f"  PASS {int(stable_df['keep'].sum())}/{len(stable_df)}  ->  {len(hyps_stable)} passed to Phase 3"
          + (f"  (incl. {len(provisional)} provisional)" if provisional else ""))
    for _, r in stable_df.sort_values('selection_frequency', ascending=False).iterrows():
        nf = int(r['n_fire']) if r['n_fire'] == r['n_fire'] else 0
        sf = r['selection_frequency'] if r['selection_frequency'] == r['selection_frequency'] else 0.0
        qv = r['qvalue'] if r['qvalue'] == r['qvalue'] else float('nan')
        tag = 'KEEP' if r['keep'] else ('prov' if r['provisional'] else 'drop')
        print(f"  [{tag}] fire={nf:>4} uni={r['univariate_coef']:+.2f} q={qv:.3f} selfreq={sf:.2f}  "
              f"{str(r['hypothesis'])[:50]}")
print(f'Phase 2.5 complete in {_time.time()-_t0:.1f}s')


## Phase 3: interpretable prediction
The certified hypotheses are annotated on the full training and test sets, and the binary matrix feeds an L2-regularized logistic regression with class-balanced weights. The regularization strength is chosen from ten values by cross-validated AUROC on the training set. AUROC and F1 on the held-out test set are reported with 95% percentile intervals from `EVAL_N_BOOT` bootstrap resamples, and F1 uses the prevalence-matched operating point. Every prediction is a weighted sum over named concepts.

The certified library lists each hypothesis with its coefficient in this model, its prevalence in the training annotations and its demographic tag. Significance markers come from a separate unregularized logistic model on the same training annotations, with Benjamini-Hochberg control at q = 0.10; that model does not affect predictions or metrics.

Output: `test_predictions.csv`, `certified_library.csv` and `hypothesis_lineage.csv`, which records the fate of every candidate (certified, merged or dropped at a named stage). With `EXPORT_PREDICTIONS`, the predictions are also written to `predictions/<task>/clip_h/<arm>.csv`.


In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, f1_score
from hypothesaes.annotate import annotate_texts_with_concepts
import time as _time

print('='*60); print('PHASE 3  |  Annotate train and test -> L2 logistic head on the certified library'); print('='*60)
_t0 = _time.time()

hyps = list(hyps_stable)
print(f'Annotating {len(train_profiles)} train x {len(hyps)} hypotheses...')
_MSM = globals().get('MERGE_SOURCE_MAP', {})
def _fit_ann_tr(_t):
    return annotate_texts_with_concepts(texts=train_profiles, concepts=list(_t),
        cache_name=CACHE_NAME + '_train_final', model=ANNOT_MODEL, n_workers=ANNOT_WORKERS, seed=SEED,
        use_batch=USE_BATCH_ANNOTATION)
train_ann = L.annotate_features(train_profiles, hyps, _MSM, _fit_ann_tr)
print(f'Annotating {len(test_profiles)} test profiles...')
def _fit_ann_te(_t):
    return annotate_texts_with_concepts(texts=test_profiles, concepts=list(_t),
        cache_name=CACHE_NAME + '_test', model=ANNOT_MODEL, n_workers=ANNOT_WORKERS, seed=SEED,
        use_batch=USE_BATCH_ANNOTATION)
test_ann = L.annotate_features(test_profiles, hyps, _MSM, _fit_ann_te)

prev = {h: float(np.mean(train_ann[h] != 0)) for h in hyps}
_dm = (hypotheses_df.set_index('interpretation')['is_demographic'].to_dict()
       if 'is_demographic' in hypotheses_df.columns else {})
def _isdemo(h): return bool(_dm.get(h, False))
all_hyps = list(hyps)
print(f'  {sum(not _isdemo(h) for h in all_hyps)} clinical | {sum(_isdemo(h) for h in all_hyps)} demographic | '
      f'{len(all_hyps)} certified')

X_tr_full = np.column_stack([train_ann[h] for h in all_hyps]).astype(float)
X_te_full = np.column_stack([test_ann[h] for h in all_hyps]).astype(float)

# L2 head with class-balanced weights; C is chosen from ten values by cross-validated AUROC.
_cv = max(2, min(5, int(min(np.bincount(y_train)))))
clf = LogisticRegressionCV(Cs=10, cv=_cv, max_iter=4000, class_weight='balanced',
                           scoring='roc_auc', random_state=SEED)
clf.fit(X_tr_full, y_train)
p_te = clf.predict_proba(X_te_full)[:, 1]; p_tr_full = clf.predict_proba(X_tr_full)[:, 1]
# Prior correction (Elkan, 2001; King and Zeng, 2001) maps the scores from the training prevalence
# to the test prevalence. With both sets balanced the two prevalences are equal and the scores are
# unchanged apart from clipping to [1e-6, 1 - 1e-6].
_prev_tr = float(np.mean(y_train)); _prev_te = float(np.mean(y_test))
p_te_eval = L.prevalence_correct(p_te, _prev_tr, _prev_te) if TRAIN_BALANCE is not None else p_te

# F1 operating point (see F1_THRESHOLD_MODE in Step 1). The train-tuned threshold is recorded in
# every mode and used in 'train_tuned' mode.
p_tr_eval = L.prevalence_correct(p_tr_full, _prev_tr, _prev_te) if TRAIN_BALANCE is not None else p_tr_full
THR_TUNED_REF = L.tuned_threshold(y_train, p_tr_eval)
if F1_THRESHOLD_MODE == 'prevalence_matched':
    _yhat_te = L.prevalence_matched(y_test, p_te_eval, seed=SEED)
    THR_FIT = float('nan')                              # top-k has no single threshold
else:
    THR_FIT = THR_TUNED_REF if F1_THRESHOLD_MODE == 'train_tuned' else 0.5
    _yhat_te = (p_te_eval >= THR_FIT).astype(int)
fit_metrics = {'auroc': roc_auc_score(y_test, p_te),
               'f1': f1_score(y_test, _yhat_te, zero_division=0),
               'decision_threshold': float(THR_FIT),
               'C_selected': float(clf.C_[0])}
# One set of bootstrap resamples serves AUROC and F1, and the feature baselines below reuse the
# same draws, so the intervals and any paired differences are comparable.
_boot_groups = (test_df['subject_id'].to_numpy()
                if (EVAL_RESAMPLE_BY == 'subject' and 'subject_id' in test_df.columns) else None)
EVAL_RESAMPLES = L.make_resamples(len(y_test), n_boot=EVAL_N_BOOT, seed=SEED, groups=_boot_groups)
ci = L.bootstrap_ci(y_test, p_te, metric='auroc', seed=SEED, alpha=EVAL_ALPHA,
                    resamples=EVAL_RESAMPLES)
# The F1 interval comes from the same scorer as the feature baselines.
_sc = L.score_with_ci(y_test, p_te_eval, pred=_yhat_te, n_boot=EVAL_N_BOOT, seed=SEED)
ci_f1 = {'point': _sc['f1'] / 100.0, 'lo': _sc['f1_ci'][0] / 100.0,
         'hi': _sc['f1_ci'][1] / 100.0, 'half_width': _sc['f1_pm'] / 100.0}
# The exported scores are the prior-corrected ones; confirm that the correction kept the ranking.
_auroc_exported = roc_auc_score(y_test, p_te_eval)
assert abs(_auroc_exported - fit_metrics['auroc']) < 1e-9, (
    f"prevalence_correct changed the ranking: AUROC(p_te)={fit_metrics['auroc']:.10f} vs "
    f"AUROC(p_te_eval)={_auroc_exported:.10f}")
fit_metrics['auroc_ci'] = [ci['lo'], ci['hi']]
fit_metrics['auroc_half_width'] = ci['half_width']
fit_metrics['f1_ci'] = [ci_f1['lo'], ci_f1['hi']]
fit_metrics['f1_half_width'] = ci_f1['half_width']

# Test-set predictions.
_pid_te = (test_df['hadm_id'] if 'hadm_id' in test_df.columns else pd.Series(np.arange(len(y_test)))).to_numpy()
_sid_te = test_df['subject_id'].to_numpy() if 'subject_id' in test_df.columns else None
pd.DataFrame({'patient_id': _pid_te, 'subject_id': (_sid_te if _sid_te is not None else _pid_te),
              'y_true': np.asarray(y_test).astype(int), 'y_score': p_te,
              'y_score_prior_corrected': p_te_eval,
              'y_pred': _yhat_te}).to_csv(os.path.join(RUN_DIR, 'test_predictions.csv'), index=False)
if EXPORT_PREDICTIONS:
    L.write_predictions(PRED_ROOT, TASK, 'clip_h', GENERATOR, _pid_te, y_test,
                        y_score=p_te_eval, y_pred=_yhat_te, subject_ids=_sid_te)
    L.record_threshold(PRED_ROOT, TASK, f'clip_h/{GENERATOR}', float(THR_FIT))
    print(f'  exported predictions -> {PRED_ROOT}')

print(f"\n--- Phase 3 held-out results ({len(all_hyps)} certified hypotheses, C={fit_metrics['C_selected']:.3g}) ---")
print(f"  AUROC {fit_metrics['auroc']:.3f} (95% CI {ci['lo']:.3f}-{ci['hi']:.3f})")
print(f"  F1    {fit_metrics['f1']:.3f} (95% CI {ci_f1['lo']:.3f}-{ci_f1['hi']:.3f}, operating point: {F1_THRESHOLD_MODE})")

coefs = clf.coef_[0]
# Descriptive refit: an unregularized logistic model on the same training annotations gives Wald
# p-values, adjusted with Benjamini-Hochberg at q = 0.10. It is reported with the library only.
reg_inf = L.logit_inference(X_tr_full, y_train, all_hyps, seed=SEED).set_index('name')
_reg_p = [float(reg_inf.loc[h, 'pvalue']) for h in all_hyps]
_rh_rej, _rh_q = L.bh_fdr(_reg_p, q=0.10)
reg_coef_map = {h: float(reg_inf.loc[h, 'coef']) for h in all_hyps}
reg_p_map = dict(zip(all_hyps, _reg_p))
reg_bhq_map = dict(zip(all_hyps, _rh_q)); reg_bhsig_map = dict(zip(all_hyps, _rh_rej))
# Gate statistics from Phase 2.5 and the critique fields from Phase 2.3.
selfreq_map = dict(zip(stable_df['hypothesis'], stable_df['selection_frequency']))
q_map = dict(zip(stable_df['hypothesis'], stable_df['qvalue']))
dir_map = hypotheses_df.set_index('interpretation')['expected_direction'].to_dict() if 'expected_direction' in hypotheses_df.columns else {}
mech_map = hypotheses_df.set_index('interpretation')['mechanism'].to_dict() if 'mechanism' in hypotheses_df.columns else {}

rows = []
for i, h in enumerate(all_hyps):
    rows.append({'hypothesis': h, 'regression_coef': float(coefs[i]), 'feature_prevalence': prev[h],
                 'is_demographic': _isdemo(h),
                 'reg_coef': float(reg_coef_map.get(h, float('nan'))), 'reg_p': float(reg_p_map.get(h, float('nan'))),
                 'reg_q_bh': float(reg_bhq_map.get(h, float('nan'))), 'sig_bh': bool(reg_bhsig_map.get(h, False)),
                 'gate_q': float(q_map.get(h, float('nan'))),
                 'selection_frequency': float(selfreq_map.get(h, float('nan'))),
                 'expected_direction': str(dir_map.get(h, 'unclear')).lower(),
                 'mechanism': str(mech_map.get(h, ''))})
library = pd.DataFrame(rows).sort_values('regression_coef', ascending=False).reset_index(drop=True)
library.to_csv(os.path.join(RUN_DIR, 'certified_library.csv'), index=False)
_nbh = int(library['sig_bh'].sum())
significance = {'n_tested': len(all_hyps), 'q': 0.10, 'n_sig_bh': _nbh}
fit_metrics['n_sig_bh'] = _nbh

print('\n-- Certified library: coefficient, training prevalence, BH significance in the descriptive refit (*) --')
for _, r in library.iterrows():
    print(f"  {r['regression_coef']:+.3f}  {r['feature_prevalence']:6.1%}  {'*' if r['sig_bh'] else ' '}"
          f"{' [demographic]' if r['is_demographic'] else ''}  {str(r['hypothesis'])[:70]}")
print(f'  {_nbh}/{len(all_hyps)} significant under BH (q <= 0.10) in the descriptive refit')

# Lineage: the fate of every screened candidate (certified, merged, or dropped at a named stage).
try:
    _slog = pd.read_csv(os.path.join(RUN_DIR, 'stage_screen.csv'))
    # Neuron provenance is carried where the text is unchanged through refinement, and
    # DEDUP_RESULT supplies the complement drops, so those are reported as dropped at 2.4.
    _cands = L.lineage_candidates_from_stage_logs(RUN_DIR)
    _sstatus = {str(r['hypothesis']): ('kept' if bool(r['kept']) else f"dropped:{r['reason']}")
                for _, r in _slog.iterrows()}
    _keptset = set(hyps_stable)   # everything passed to Phase 3, including provisional hypotheses
    _gstatus = {h: ('certified' if h in _keptset else 'dropped:gate') for h in stable_df['hypothesis']}
    _coefs = dict(zip(library['hypothesis'], library['regression_coef']))
    _dedup = globals().get('DEDUP_RESULT', {'source_map': globals().get('MERGE_SOURCE_MAP', {})})
    _lineage = L.build_lineage(_cands, _sstatus, _dedup, _gstatus, _coefs)
    L.log_stage_table(RUN_DIR, 'hypothesis_lineage', _lineage)
    _nc = sum(1 for r in _lineage if r['final_status'] == 'certified')
    _nm = sum(1 for r in _lineage if str(r['final_status']).startswith('merged_into'))
    _nd = sum(1 for r in _lineage if 'dropped' in str(r['final_status']))
    print(f'  lineage: {_nc} certified, {_nm} merged, {_nd} dropped  ->  hypothesis_lineage.csv')
except Exception as _e:
    print(f'  [warn] lineage not written: {_e}')
print(f'\nPhase 3 complete in {_time.time()-_t0:.1f}s')


## Evaluation: feature baselines
Three supervised baselines are fitted on the training split and scored on the same test set, with the same bootstrap resamples and F1 operating point as CLIP-H: TF-IDF with logistic regression over the profile text, logistic regression over the structured features, and XGBoost over the same structured features. Hyperparameters are in `mimic_eval/baselines_ci.py`. `heldout_metrics.csv` holds the CLIP-H and feature-baseline rows of the main results table, with AUROC, F1 and their 95% intervals.


In [ ]:
import time as _time

print('='*60)
print('EVALUATION  |  Feature baselines: TF-IDF + LR, structured LR, XGBoost')
print('='*60)
_t0 = _time.time()

_bl = L.run_feature_baselines(train_df, test_df, train_profiles, test_profiles, y_train, y_test,
                              task=TASK, seed=SEED, resamples=EVAL_RESAMPLES, alpha=EVAL_ALPHA,
                              n_boot=EVAL_N_BOOT, threshold_mode=F1_THRESHOLD_MODE)
base_tfidf, base_lr, base_xgb = (_bl['results']['tfidf_lr'], _bl['results']['structured_lr'],
                                 _bl['results']['xgboost'])

# run_feature_baselines adds an 'error' column only when a baseline fails, and the successful
# rows then carry NaN there, so a row is valid when its error is missing, NaN or empty.
def _ok(r):
    e = r.get('error')
    return e is None or (isinstance(e, float) and e != e) or str(e).strip() == ''
baseline_table = [r.to_dict() for _, r in _bl['table'].iterrows() if _ok(r)]
for _, _r in _bl['table'].iterrows():
    if not _ok(_r):
        print(f"  [warn] {_r['method']} unavailable: {_r['error']}")
baseline_table.append({'method': 'clip_h', 'auroc': fit_metrics['auroc'],
                       'auroc_lo': ci['lo'], 'auroc_hi': ci['hi'],
                       'auroc_half_width': ci['half_width'], 'f1': fit_metrics['f1'],
                       'f1_lo': ci_f1['lo'], 'f1_hi': ci_f1['hi'],
                       'f1_half_width': ci_f1['half_width'], 'threshold': float(THR_FIT)})
baseline_df = pd.DataFrame(baseline_table)
baseline_df.to_csv(os.path.join(RUN_DIR, 'heldout_metrics.csv'), index=False)

print(f'\n--- Held-out comparison (same split, same {EVAL_N_BOOT} bootstrap resamples, '
      f'F1 operating point: {F1_THRESHOLD_MODE}) ---')
for _r in baseline_table:
    print(f"  {_r['method']:<14} AUROC {100*_r['auroc']:5.2f} (+-{100*_r['auroc_half_width']:.2f})   "
          f"F1 {100*_r['f1']:5.2f} (+-{100*_r['f1_half_width']:.2f})")
print(f'\nBaselines complete in {_time.time()-_t0:.1f}s')


## Run record
`manifest.json` in `RUN_DIR` records the settings, models, package versions, cohort sizes, gate counts, held-out metrics and data-quality notes. An annotation call that fails after all retries enters the feature matrix as "absent", so the number of such failures is counted, kept across kernel restarts and reported here. Failed calls are not cached, so rerunning the affected phase fetches them again.


In [ ]:
import json, time as _time
from hypothesaes import annotate as _annot_mod
# Annotations that failed after all retries were recorded as 0 ('concept absent'). The count is
# kept in annotation_failures.json, so it survives a kernel restart, and written to the manifest.
_afp = os.path.join(RUN_DIR, 'annotation_failures.json')
_prev_fail = 0
if os.path.exists(_afp):
    try:
        _prev_fail = int(json.load(open(_afp, encoding='utf-8')).get('total', 0))
    except Exception:
        _prev_fail = 0
N_ANNOTATION_FAILURES = max(int(getattr(_annot_mod, 'ANNOTATION_FAILURES', 0)), _prev_fail)
json.dump({'total': int(N_ANNOTATION_FAILURES), 'this_session':
           int(getattr(_annot_mod, 'ANNOTATION_FAILURES', 0))},
          open(_afp, 'w', encoding='utf-8'))

print('='*60)
print('RUN RECORD  |  manifest and summary')
print('='*60)

manifest = {
    'task': TASK, 'seed': SEED,
    'models': {'interp': INTERP_MODEL, 'annot': ANNOT_MODEL, 'critic': CRITIC_MODEL,
               'dedup': DEDUP_MODEL,
               'claude': (CLAUDE_MODEL if CRITIC_PROVIDER in ('anthropic', 'both') else None)},
    'emb_model': EMB_MODEL, 'selection_method': SELECTION_METHOD,
    'use_batch_annotation': bool(USE_BATCH_ANNOTATION),   # batch and synchronous annotation give the same outputs
    'n_train': int(len(train_df)), 'n_test': int(len(test_df)),
    'n_candidate_hypotheses': int(N_HYPOTHESES),
    'n_hypotheses_final': int(len(hypotheses_df)),
    'critic': {'n_final': int(len(hypotheses_df)),
               'n_demographic': int(hypotheses_df['is_demographic'].sum()) if 'is_demographic' in hypotheses_df.columns else 0,
               'mean_validity': float(hypotheses_df['clinical_validity'].mean()) if 'clinical_validity' in hypotheses_df.columns else None,
               'n_circularity_flagged': int((hypotheses_df['leakage_risk'] == 1).sum()) if 'leakage_risk' in hypotheses_df.columns else 0,
               'ensemble_kappa': (ensemble_kappa if 'ensemble_kappa' in dir() else None)},
    'gating': {'min_fire_count': int(globals().get('GATE_MIN_FIRE', ESTIMABILITY_FLOOR)),
               'n_stable': int(stable_df['keep'].fillna(False).sum()),
               'n_fed_to_fit': int(len(all_hyps)), 'n_provisional': int(stable_df['provisional'].sum()),
               'fdr_q': FDR_Q, 'stability_pi': STABILITY_PI, 'stability_b': STABILITY_B},
    'significance': significance,
    'fit_metrics': fit_metrics, 'auroc_ci': ci,
    'baselines': {'tfidf_lr': base_tfidf['auroc'], 'structured_lr': base_lr['auroc'],
                  'xgboost': base_xgb['auroc']},
    'heldout_table': baseline_df.to_dict('records'),
    'eval_protocol': {'n_boot': int(EVAL_N_BOOT), 'alpha': float(EVAL_ALPHA),
                      'resample_by': EVAL_RESAMPLE_BY,
                      'f1_threshold_mode': F1_THRESHOLD_MODE,
                      'decision_threshold': float(THR_FIT),
                      'decision_threshold_train_tuned_ref': float(THR_TUNED_REF)},
    'annotation_failures': int(N_ANNOTATION_FAILURES),
    'note': 'Hosted models at temperature 0 with a seed are close to, but not exactly, deterministic.',
}
# Data-quality issues are recorded in the manifest and printed; they do not stop the run.
_issues = []
_nfail = int(N_ANNOTATION_FAILURES)
if _nfail:
    _issues.append(
        f"{_nfail} annotation call(s) failed and were recorded as absent "
        f"(this session {int(getattr(_annot_mod, 'ANNOTATION_FAILURES', 0))}, earlier sessions "
        f"{_prev_fail}). Failed calls are not cached: delete the affected phase's checkpoint and "
        f"rerun it, then delete {_afp} to reset the count.")
manifest['data_quality'] = {'clean': not _issues, 'issues': _issues,
                            'annotation_failures': _nfail}
path = L.write_manifest(RUN_DIR, manifest)
print('Manifest written:', path)
if _issues:
    print('')
    print('Data-quality issues (recorded under data_quality in manifest.json):')
    for _i in _issues:
        print('  * ' + _i)

total_mins = (_time.time() - _PIPELINE_START) / 60
print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)
print(f"  Task              : {TASK}")
print(f"  Train / Test      : {manifest['n_train']} / {manifest['n_test']}")
print(f"  Certified library : {len(all_hyps)} hypotheses")
print(f"  CLIP-H AUROC      : {fit_metrics['auroc']:.3f}  (95% CI {ci['lo']:.3f}-{ci['hi']:.3f})")
print(f"  CLIP-H F1         : {fit_metrics['f1']:.3f}  (95% CI {ci_f1['lo']:.3f}-{ci_f1['hi']:.3f})")
print(f"  TF-IDF + LR AUROC : {base_tfidf['auroc']:.3f}")
print(f"  LR AUROC          : {base_lr['auroc']:.3f}")
print(f"  XGBoost AUROC     : {base_xgb['auroc']:.3f}")
print(f"  Total runtime     : {total_mins:.1f} min")
print('='*60)
print(f'\nResults saved to: {RUN_DIR}')
